In [ ]:
import datetime
import re
from functools import reduce

import numpy as np
import pandas as pd
from keyring import get_password
from redcap import Project
from tqdm import tqdm

from redcap_aux import (
    CCP,
    COL_OVERLAPS,
    DATE_TRIPLET_COLS,
    DURATION_MY,
    LAB_ID,
    MULTIINDICES,
    RACE_MAP,
    RENAME_MAPPER,
    REPLACE_DATES,
    REPLACEMENTS,
    SLP,
    fix_monthsontbtx,
    get_date_triplet,
    get_transform,
    idxmax,
    merge,
    sl_int,
    standardise_treatment,
    fillna_same,
)

## REDCap

### Extract

In [ ]:
PROJECT_NAMES = [SLP, CCP]
REDCAP_URL = get_password("redcap", "URL")
api_key = {}
project = {}
recs = {}
first_inst = {}
metadata = {}
KWS = dict(raw_or_label="label", df_kwargs=dict(low_memory=False, index_col=False))
for name in PROJECT_NAMES:
    api_key[name] = get_password("redcap", name)
    project[name] = Project(REDCAP_URL, api_key[name])
    recs[name] = project[name].export_records("df", **KWS).dropna(axis=1, how="all")
    first_inst[name] = project[name].export_instruments()[0]["instrument_label"]
    # Create a mapping dictionary for checkbox fields
    metadata[name] = pd.DataFrame(project[name].export_metadata())

### Transform

In [ ]:
########### SLP Data Cleaning ###########
slp = recs[SLP].copy()
# Fix errors
slp["age_main"] = slp.age_main.replace({"3m 20d": "0.3048", ">70": pd.NA, "20s": pd.NA})
slp["date_of_birth"] = slp.date_of_birth.replace(REPLACE_DATES)
# Value mapping and type coercion
slp["yob"] = pd.to_numeric(slp.date_of_birth, "coerce")
slp.loc[slp.yob.notna(), "date_of_birth"] = pd.NA
slp["sl"] = reduce(lambda s1, s2: s1.fillna(s2), (slp[c] for c in LAB_ID)).map(sl_int)
slp = slp.apply(get_transform)
slp["treatment"] = slp["treatment"].map(standardise_treatment)
slp["redcap_repeat_instrument"] = slp.redcap_repeat_instrument.fillna(first_inst[SLP])
slp["gender_main"] = slp.gender_main.replace("Not specified", pd.NA)
slp["year_collected"] = slp.year_collected.fillna(slp.sample_collection_date.dt.year)
# Semantic cleaning and validation
# Coalesce redundant fields
slp["race"] = slp.race_main.fillna(slp.race_main_2).map(RACE_MAP)
slp["yob"] = slp.yob.fillna(slp.year_collected - slp.age_main - 0.5)
slp["age_main"] = slp.age_main.fillna(slp.year_collected - slp.yob - 0.5)


########### CCP Data Cleaning ###########
ccp = recs[CCP].copy()
# Fix errors
ccp["monthsontbtx"] = ccp.monthsontbtx.map(fix_monthsontbtx)
for col, repl in REPLACEMENTS.items():
    ccp[col] = ccp[col].replace(repl)
# Value mapping and type coercion
ccp["redcap_repeat_instrument"] = ccp.redcap_repeat_instrument.fillna(first_inst[CCP])
ccp_na = ["U", "UU", "uu", "Uu", "unknown", "u", "nk", "D", "A", "Nk", "UK"]
ccp = ccp.replace(ccp_na, pd.NA)
for col, ymd in DATE_TRIPLET_COLS.items():
    ccp[col] = get_date_triplet(ccp[ymd])
for yr_col, mnth_col in DURATION_MY.items():
    ccp.loc[ccp[yr_col] == ccp[mnth_col] / 12, mnth_col] = 0
    ccp[yr_col] = ccp[yr_col] + ccp[mnth_col].fillna(0) / 12
ccp = ccp.apply(get_transform)
# Semantic cleaning and validation
# Coalesce duplicate fields
ccp["race"] = ccp.race.fillna(ccp.spcfyrace)
ccp["age"] = ccp.age.fillna(np.floor((ccp.compdatedem - ccp.dob).dt.days / 365.25))


########### Parallel Standardisation ###########
PATTERN = r"obo[:/]NCIT_C\d{4,}|:|\(|\)|/|–|—|\+|&|-"
for name, df in zip([SLP, CCP], [slp, ccp]):
    # Create a mapping dictionary for checkbox fields
    checkbox_rename = {}
    for __, row in metadata[name][metadata[name].field_type == "checkbox"].iterrows():
        field = row["field_name"]
        choices = row["select_choices_or_calculations"].split("|")
        raw_rename = {}
        for choice in choices:
            code, label = choice.strip().split(", ", 1)
            raw_rename[f"{field}___{code}"] = label
        if df[raw_rename.keys()].sum(axis=1).max() == 1:
            # Mutually exclusive: collapse to single column
            df[field] = df[raw_rename.keys()].apply(idxmax, axis=1).map(raw_rename)
            df = df.drop(columns=raw_rename.keys())
        else:
            # Not mutually exclusive: rename columns
            for code, label in raw_rename.items():
                clean = re.sub(PATTERN, "", label).strip().lower().replace(" ", "_")
                checkbox_rename[code] = f"{field}___{clean}"
    df = df.rename(columns=checkbox_rename).convert_dtypes()
    # Separate out REDCap repeat instruments
    for inst in df.redcap_repeat_instrument.unique():
        recs[inst] = (
            df[df.redcap_repeat_instrument == inst]
            .convert_dtypes()
            .rename(columns=RENAME_MAPPER)
            .drop_duplicates()
            .dropna(how="all")
            .dropna(how="all", axis=1)
            .set_index(MULTIINDICES[inst])
        )
by_idx = {idx: {fi: recs[fi]} for idx, fi in zip(["sl", "pid"], first_inst.values())}
for key, df in recs.items():
    if len(df.index.names) > 1:
        gb = df.groupby(df.index.names[0])
        by_idx[df.index.names[0]][key] = gb.first().loc[:, gb.nunique().max() == 1]
for idx, value in by_idx.items():
    by_idx[idx] = reduce(lambda kv1, kv2: merge(kv1, kv2, idx), value.items())[1]
sldf_full = by_idx["sl"].merge(
    by_idx["pid"], how="left", on="pid", suffixes=("_sl", "_pid")
)
# sldf = sldf_full.drop(columns=drop)

In [ ]:
dfc = sldf_full[
    [
        "sample_collection_date",
        "autopsy",
        "nature_of_death",
        "cause_of_death",
        "treatment",
        "epilepsy",
        "hypertension",
        "is_block_ctscanned",
        "vdate",
        "datetoldtb",
        "tbtxstartdate",
        "proptbdse",
        "lungtssuedate",
        "xraydate",
        "xrayres",
        "pohivdate",
        "neghivdate",
        "artstartdate",
        "weight",
        "height",
        "compdatelung",
    ]
].copy()
come_back_to = [
    "other_source",
    "specimen_source",
    "history_1",
    "patient_other_comments",
    "other_comorbility",
    ["tb", "arm___case"],
    "tbtype",
    [
        "external_id",
        "external_id2",
        "rs_id",
        "pid",
        "altpid",
    ],
]
dfc_cols = [
    ["race_sl", "race_pid", "spcfyrace"],
    ["gender", "gender_main"],
    ["dob", "date_of_birth"],
    ["hiv", "hivstatus"],
    ["asthma", "ongastm"],
    ["copd", "ongcopd"],
    ["bronchiectasis", "bronch", "ongbro"],
    ["diabetes_sl", "diabetes_pid", "ongdiab"],
    ["cancer", "ongcncr"],
]
for sources in dfc_cols:
    dfc[sources[0].removesuffix("_sl")] = sldf_full[sources].apply(fillna_same, axis=1)

dfc["year_collected"] = pd.concat(
    [
        sldf_full.year_collected,
        sldf_full.sample_collection_date.dt.year,
        sldf_full.lungtssuedate.dt.year,
        sldf_full.vdate.dt.year,
        sldf_full.compdatelung.dt.year,
        sldf_full.proptbdse.dt.year,
    ],
    axis=1,
).apply(fillna_same, axis=1, tol=1)
dfc["age"] = (
    sldf_full[["age_main", "age"]]
    .replace({"age": {0: pd.NA}})
    .apply(fillna_same, axis=1, tol=1)
)
dfc["yob"] = pd.concat(
    [sldf_full.date_of_birth.dt.year, sldf_full.dob.dt.year, sldf_full.yob], axis=1
).apply(fillna_same, axis=1, tol=0.5, aggfunc=lambda s: s.dropna().iloc[0])

#### Separate out REDCap instruments

In [ ]:
pef = (
    slp[slp.redcap_repeat_instrument == "Participants Entry Form"]
    .set_index("sl")
    .dropna(how="all", axis=1)
)

In [ ]:
pef.loc[early_sls].dropna(axis=1, how="all")[
    [
        "sample_collection_date",
        "year_collected",
        "autopsy",
        "scan_participant_id",
        "gender_main",
        "age_main",
        "history_1",
        "treatment",
        "patient_other_comments",
        "comorbidity___1",
        "comorbidity___7",
        "other_comorbility",
        "hiv",
        "yob",
    ]
].to_clipboard(index=False)

#### Merging

In [ ]:
# Alignment checks

# Merge
sldf = recs["Participants Entry Form"].merge(
    recs["Baseline"], how="left", on="pid", validate="many_to_one"
)

# Reconciliation
for key, value in FILL_NA.items():
    sldf[key] = sldf[key].fillna(sldf[value])
sldf["dob"] = sldf.dob.fillna(
    pd.to_datetime(sldf.date_of_birth, "coerce", format="%Y-%m-%d")
)
sldf["date_of_birth"] = sldf.dob.dt.year.fillna(
    pd.to_numeric(sldf.date_of_birth, "coerce")
)
for keep, drop in COL_OVERLAPS.items():
    sldf[keep] = sldf[keep].fillna(sldf[drop])
    sldf.loc[sldf[keep] != sldf[drop], keep] = pd.NA
sldf["year_collected"] = sldf.year_collected.fillna(sldf.imp_tissue_date.dt.year)
sldf["date_of_birth"] = sldf.date_of_birth.fillna(sldf.dob.dt.year)
sldf["imp_tbtxstartdate"] = sldf.tbtxstartdate.fillna(sldf.datetoldtb).fillna(
    sldf.pid.map(recs["TB Medications"].groupby("pid")["tbmed_medstartdate"].min())
)
# Imputation

# Derived variables
sldf["imp_tissue_date"] = (
    sldf.lungtssuedate.fillna(sldf.proptbdse + datetime.timedelta(days=1))
    .fillna(sldf.vdate + datetime.timedelta(days=1))
    .fillna(sldf.sample_collection_date)
)
sldf["imp_treatment_duration"] = (sldf.imp_tissue_date - sldf.imp_tbtxstartdate).dt.days


### Load

### Early Lesions

In [ ]:
early_manifest = pd.read_csv("early_manifest.csv", index_col=[0, 1]).convert_dtypes()
early_manifest.loc[early_manifest.index.duplicated(keep=False)] = (
    early_manifest.loc[early_manifest.index.duplicated(keep=False)]
    .groupby(level=[0, 1])
    .sum(min_count=1)
)
early_manifest = (
    early_manifest[~early_manifest.index.duplicated()]
    .sort_index()
    .merge(
        dfc[["autopsy", "year_collected", "hiv", "gender", "age", ""]],
        how="left",
        left_on="sl",
        right_index=True,
    )
)
early_manifest.to_csv("early_manifest2.csv")

## OMERO

In [ ]:
from keyring import get_password
from omero.gateway import BlitzGateway

OME_CALL = {k: get_password("omero", k) for k in ["username", "passwd", "host", "port"]}
IV_DS = f"https://{OME_CALL['host'].replace('omero', 'microscopy')}/iviewer/?dataset="

SKIP = ["test", "ZN Dataset"]

conn = BlitzGateway(**OME_CALL)
conn.connect()

# Your existing code for projects/datasets
omero_sls = {}
prefixes = {key: key.split()[0].casefold() for key in ["WSI", "Macro Images"]}


def search_filenames(conn, substring="early"):
    substring = substring.lower()
    matches = []
    for img in tqdm(conn.getObjects("Image"), desc="Images"):
        if substring in img.getName().lower():
            matches.append(img)
    return matches


# Usage
early_images = search_filenames(conn, "early")

In [ ]:
matches = {
    name: search_filenames(conn, name)
    for name in [" [macro mask image]", " [macro image]", " [0]", ".ndpi"]
}

In [ ]:
def crawl(path):
    for entry in os.scandir(path):
        if entry.is_dir():
            yield from crawl(entry.path)
        else:
            yield entry


annotations = []
it = tqdm(crawl("/Volumes/Imaging/AllSLs"))
for file in it:
    if file.name.endswith(".ndpa"):
        annotations.append(file.path)
        it.set_description(file.path)

In [ ]:
annotated = {
    "early": [],
    "tyler": [],
    "error": [],
}

for path in tqdm(annotations):
    with open(path, "r") as f:
        try:
            s = f.read().lower()
        except (UnicodeDecodeError, OSError):
            annotated["error"].append(path)
            continue
        if "early" in s:
            annotated["early"].append(path)
        if "tyler" in s:
            annotated["tyler"].append(path)